# Kafka + Spark Structured Streaming — NYC Taxi Ride Analytics



## Pipeline Architecture

```
 Parquet files                Kafka Broker               Spark Structured Streaming
 (historical data)                                        (real-time)

 yellow_tripdata_             topic:                      ┌──────────────────────────┐
 2021..2025.parquet  ──────▶  taxi-rides          ──────▶ │  Stateless Anomaly       │
                              3 partitions                 │  Scorer (per ride)       │
 Producer replays             key = PULocationID           └──────────┬───────────────┘
 at controlled rate           (zone-based routing)                    │
                                                           ┌──────────▼───────────────┐
                                                           │  Feature Engineering     │
                                                           │  (per zone, 5-min window)│
                                                           └──────────┬───────────────┘

---

## Kafka Concepts in this Demo

| Concept | Role here |
|---------|----------|
| **Topic** `taxi-rides` | One message per completed ride |
| **Partition** (3) | Keyed by `PULocationID` → rides from same zone in same partition → zone-level ordering |
| **Producer** | Replays parquet rows to Kafka at controlled rate (simulates live dispatch system) |
| **Consumer group** | `taxi-analytics-group` — Spark shares partitions across executors |
| **Offset / checkpoint** | Spark commits offsets — supports exactly-once feature computation |
| **Event replay** | Re-read from `earliest` to recompute features for any historical window (backfill for model retraining) |


---
## 1 — Install Dependencies

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kafka-python-ng'])
print('Dependencies ready.')


---
## 2 — Start Kafka (Docker)

```bash
docker-compose up -d
docker-compose ps     # kafka-taxi-demo should be running
```


In [ ]:
compose = '''\
services:
  kafka:
    image: apache/kafka:3.9.0
    container_name: kafka-taxi-demo
    ports:
      - "9092:9092"
    environment:
      KAFKA_NODE_ID: 1
      KAFKA_PROCESS_ROLES: broker,controller
      KAFKA_LISTENERS: PLAINTEXT://0.0.0.0:9092,CONTROLLER://127.0.0.1:9093
      KAFKA_ADVERTISED_LISTENERS: PLAINTEXT://localhost:9092
      KAFKA_INTER_BROKER_LISTENER_NAME: PLAINTEXT
      KAFKA_CONTROLLER_LISTENER_NAMES: CONTROLLER
      KAFKA_LISTENER_SECURITY_PROTOCOL_MAP: PLAINTEXT:PLAINTEXT,CONTROLLER:PLAINTEXT
      KAFKA_CONTROLLER_QUORUM_VOTERS: 1@127.0.0.1:9093
      CLUSTER_ID: MkU3OEVBNTcwNTJENDM2Qk
      KAFKA_AUTO_CREATE_TOPICS_ENABLE: "true"
      KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR: 1
      KAFKA_NUM_PARTITIONS: 3
      KAFKA_LOG_DIRS: /tmp/kraft-logs
'''
with open('docker-compose.yml', 'w') as f:
    f.write(compose)
print('docker-compose.yml written.')
print('Run:  docker-compose down -v && docker-compose up -d')


---
## 3 — Configuration & Anomaly Rules

Each Kafka message is one completed taxi ride:
```json
{
  "ride_id": "uuid",
  "vendor_id": 2,
  "pickup_datetime": "2024-01-01T00:57:55",
  "dropoff_datetime": "2024-01-01T01:17:43",
  "passenger_count": 1,
  "trip_distance": 1.72,
  "pu_location_id": 186,
  "do_location_id": 79,
  "payment_type": 2,
  "fare_amount": 17.70,
  "tip_amount": 0.00,
  "total_amount": 22.70,
  "rate_code_id": 1
}
```

**Anomaly Rules (score ≥ 3 → flag for review):**

| # | Rule | Signal | Weight |
|---|------|--------|--------|
| A1 | Overcharge | fare/mile > $10 (non-airport) | 3 |
| A2 | Trip manipulation | `trip_duration_min > 90` | 2 |
| A3 | Ghost ride | `trip_distance < 0.1` but `fare > $15` | 3 |
| A4 | Late-night surge | `hour < 5` and `fare > $80` | 1 |
| A5 | Passenger overload | `passenger_count > 6` | 2 |


In [ ]:
KAFKA_BROKER          = 'localhost:9092'
TAXI_TOPIC            = 'taxi-rides'
CHECKPOINT_DIR        = '/tmp/spark-taxi-checkpoint'
FEATURE_STORE_PATH    = '/tmp/taxi-feature-store/zone_features'

PARQUET_PATH          = '/home/alka/MLOPs-Class/spark_demo/yellow_tripdata_2024-01.parquet'
PRODUCER_BATCH_SIZE   = 2000   # rows to read from parquet per producer run
TX_PER_SEC            = 5 #20     # publish rate, rides per second

ANOMALY_WEIGHTS = {
    'overcharge'       : 3,
    'trip_manipulation': 2,
    'ghost_ride'       : 3,
    'late_night_surge' : 1,
    'passenger_overload': 2,
}
ANOMALY_THRESHOLD = 3


print(f'Topic         : {TAXI_TOPIC}')
print(f'Parquet source: {PARQUET_PATH}')
print(f'Feature store : {FEATURE_STORE_PATH}')
print(f'Anomaly threshold: {ANOMALY_THRESHOLD}')


---
## 4 — Kafka Producer: Replay Parquet → Kafka

The producer here replays **real NYC taxi records** from the parquet file used in the Spark and Airflow classes.

Key design decisions:
- **Key = `PULocationID`** (pickup zone, as string) → all rides from the same zone go to the same
  partition → zone-level ordering guaranteed, enabling accurate per-zone windowed aggregations
- Records are read in chronological order (`tpep_pickup_datetime`) to preserve event-time semantics
- `dropoff_datetime` is included so Spark can compute `trip_duration_min` in the stream


In [ ]:
import json, uuid, time, threading, pandas as pd
from kafka import KafkaProducer
from kafka.errors import NoBrokersAvailable

_stop = threading.Event()


def load_taxi_records(path, n=PRODUCER_BATCH_SIZE):
    """Read n rows from parquet, clean nulls, sort by pickup time."""
    df = (
        pd.read_parquet(path, columns=[
            'VendorID','tpep_pickup_datetime','tpep_dropoff_datetime',
            'passenger_count','trip_distance','RatecodeID',
            'PULocationID','DOLocationID','payment_type',
            'fare_amount','tip_amount','total_amount',
        ])
        .dropna(subset=['fare_amount','trip_distance','passenger_count',
                        'tpep_pickup_datetime','tpep_dropoff_datetime'])
        .query('fare_amount > 0 and trip_distance >= 0 and passenger_count > 0')
        .sort_values('tpep_pickup_datetime')
        .head(n)
    )
    print(f'Loaded {len(df):,} taxi records from parquet.')
    return df


def row_to_message(row):
    """Convert a pandas row to the Kafka message dict."""
    return {
        'ride_id'          : str(uuid.uuid4()),
        'vendor_id'        : int(row['VendorID']) if pd.notna(row['VendorID']) else 1,
        'pickup_datetime'  : str(row['tpep_pickup_datetime']),
        'dropoff_datetime' : str(row['tpep_dropoff_datetime']),
        'passenger_count'  : int(row['passenger_count']),
        'trip_distance'    : float(row['trip_distance']),
        'pu_location_id'   : int(row['PULocationID']),
        'do_location_id'   : int(row['DOLocationID']),
        'payment_type'     : int(row['payment_type']) if pd.notna(row['payment_type']) else 2,
        'fare_amount'      : float(row['fare_amount']),
        'tip_amount'       : float(row['tip_amount']) if pd.notna(row['tip_amount']) else 0.0,
        'total_amount'     : float(row['total_amount']) if pd.notna(row['total_amount']) else 0.0,
        'rate_code_id'     : int(row['RatecodeID']) if pd.notna(row['RatecodeID']) else 1,
    }


# ── Verify one record ─────────────────────────────────────────────────────────
sample_df = load_taxi_records(PARQUET_PATH, n=5)
print('\nSample message:')
print(json.dumps(row_to_message(sample_df.iloc[0]), indent=2))


# Commands to Check Topics and their details
#### docker exec -it kafka-taxi-demo bash
#### /opt/kafka/bin/kafka-topics.sh --bootstrap-server localhost:9092 --list
#### /opt/kafka/bin/kafka-topics.sh --bootstrap-server localhost:9092 --describe

In [ ]:
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import TopicAlreadyExistsError, NoBrokersAvailable

def create_topic(broker, topic, num_partitions=3, replication_factor=1):
    try:
        admin = KafkaAdminClient(bootstrap_servers=broker)
    except NoBrokersAvailable:
        print(f'[create_topic] Kafka not reachable — run: docker compose up -d')
        return

    try:
        admin.create_topics([
            NewTopic(
                name=topic,
                num_partitions=num_partitions,
                replication_factor=replication_factor,
            )
        ])
        print(f'Topic "{topic}" created ({num_partitions} partitions).')
    except TopicAlreadyExistsError:
        print(f'Topic "{topic}" already exists — skipping creation.')
    finally:
        admin.close()


# 3 partitions so Spark can assign one executor per partition
create_topic(KAFKA_BROKER, TAXI_TOPIC, num_partitions=3)


In [ ]:
def run_producer():
    try:
        producer = KafkaProducer(
            bootstrap_servers=KAFKA_BROKER,
            key_serializer  =lambda k: str(k).encode(),
            value_serializer=lambda v: json.dumps(v).encode(),
            acks='all',
        )
    except NoBrokersAvailable:
        print('[Producer] Kafka not reachable — run: docker compose up -d')
        return

    records = load_taxi_records(PARQUET_PATH, n=PRODUCER_BATCH_SIZE)
    delay   = 1.0 / TX_PER_SEC

    print(f'[Producer] Replaying {len(records):,} rides to "{TAXI_TOPIC}" at {TX_PER_SEC} rides/s ...')

    for i, (_, row) in enumerate(records.iterrows()):
        if _stop.is_set():
            break
        msg = row_to_message(row)
        producer.send(
            TAXI_TOPIC,
            key=msg['pu_location_id'],   # zone-based partitioning
            value=msg,
        )
        if (i + 1) % 5 == 0:
            print(f'  [{i+1}/{len(records)}] zone={msg["pu_location_id"]} '
                  f'fare=${msg["fare_amount"]:.2f} dist={msg["trip_distance"]}mi')
        time.sleep(delay)

    producer.flush()
    producer.close()
    print('[Producer] Done.')


producer_thread = threading.Thread(target=run_producer, daemon=True)
producer_thread.start()
print('Producer replaying parquet → Kafka. Start Spark consumer below.')


---
## 5 — Spark Session


In [ ]:
import os, sys, shutil
os.environ.setdefault('PYSPARK_PYTHON', sys.executable)
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, LongType
)

KAFKA_PKG = 'org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1'

spark = (
    SparkSession.builder
    .appName('TaxiStreaming-FeatureStore')
    .master('local[*]')
    .config('spark.jars.packages', KAFKA_PKG)
    .config('spark.sql.shuffle.partitions', '4')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version} ready.')


---
## 6 — Read Stream from Kafka & Parse

The message timestamp is the **original ride pickup time** (`pickup_datetime`),
not the Kafka ingestion time. Spark uses this as event time for windowing —
matching the `tpep_pickup_datetime` field used in the Airflow feature engineering task.


In [ ]:
RIDE_SCHEMA = StructType([
    StructField('ride_id',         StringType(),  False),
    StructField('vendor_id',       IntegerType(), True),
    StructField('pickup_datetime', StringType(),  False),
    StructField('dropoff_datetime',StringType(),  False),
    StructField('passenger_count', IntegerType(), True),
    StructField('trip_distance',   DoubleType(),  False),
    StructField('pu_location_id',  IntegerType(), False),
    StructField('do_location_id',  IntegerType(), True),
    StructField('payment_type',    IntegerType(), True),
    StructField('fare_amount',     DoubleType(),  False),
    StructField('tip_amount',      DoubleType(),  True),
    StructField('total_amount',    DoubleType(),  True),
    StructField('rate_code_id',    IntegerType(), True),
])

raw = (
    spark.readStream
    .format('kafka')
    .option('kafka.bootstrap.servers', KAFKA_BROKER)
    .option('subscribe', TAXI_TOPIC)
    .option('startingOffsets', 'earliest')
    .option('groupIdPrefix', 'taxi-analytics')
    .option('failOnDataLoss', 'false')
    .load()
)

parsed = (
    raw
    .select(
        'partition', 'offset',
        F.from_json(F.col('value').cast('string'), RIDE_SCHEMA).alias('r'),
    )
    .select('partition', 'offset', 'r.*')
    # pickup_datetime arrives as a string — cast to timestamp for event-time windowing
    .withColumn('pickup_ts',  F.col('pickup_datetime').cast('timestamp'))
    .withColumn('dropoff_ts', F.col('dropoff_datetime').cast('timestamp'))
    # Derived columns for feature engineering and anomaly detection
    .withColumn('trip_duration_min',
        (F.unix_timestamp('dropoff_ts') - F.unix_timestamp('pickup_ts')) / 60
    )
    .withColumn('hour_of_day',  F.hour('pickup_ts'))
    .withColumn('day_of_week',  F.dayofweek('pickup_ts'))  # 1=Sun, 7=Sat
    .withColumn('is_airport',   F.col('rate_code_id').isin([2, 3]).cast('int'))
    .withColumn('tip_pct',
        F.when(F.col('fare_amount') > 0,
               F.round(F.col('tip_amount') / F.col('fare_amount') * 100, 2)
        ).otherwise(0.0)
    )
    # fare per mile — key signal for overcharge detection
    .withColumn('fare_per_mile',
        F.when(F.col('trip_distance') > 0,
               F.round(F.col('fare_amount') / F.col('trip_distance'), 2)
        ).otherwise(None)
    )
)

print('Parsed stream schema:')
parsed.printSchema()


---
## 7 — Stateless Per-Ride Anomaly Scoring

data quality checks done in real time.


In [ ]:
def anomaly_score_expr():
    a1_overcharge = F.when(
        (F.col('fare_per_mile') > 10) & (F.col('is_airport') == 0),
        ANOMALY_WEIGHTS['overcharge']
    ).otherwise(0)

    a2_long_trip = F.when(
        F.col('trip_duration_min') > 90,
        ANOMALY_WEIGHTS['trip_manipulation']
    ).otherwise(0)

    a3_ghost_ride = F.when(
        (F.col('trip_distance') < 0.1) & (F.col('fare_amount') > 15),
        ANOMALY_WEIGHTS['ghost_ride']
    ).otherwise(0)

    a4_surge = F.when(
        (F.col('hour_of_day') < 5) & (F.col('fare_amount') > 80),
        ANOMALY_WEIGHTS['late_night_surge']
    ).otherwise(0)

    a5_overload = F.when(
        F.col('passenger_count') > 6,
        ANOMALY_WEIGHTS['passenger_overload']
    ).otherwise(0)

    return (a1_overcharge + a2_long_trip + a3_ghost_ride + a4_surge + a5_overload).alias('anomaly_score')


scored = (
    parsed
    .withColumn('anomaly_score', anomaly_score_expr())
    .withColumn('triggered_rules', F.filter(
        F.array(
            F.when((F.col('fare_per_mile') > 10) & (F.col('is_airport') == 0),
                   F.lit('OVERCHARGE')),
            F.when(F.col('trip_duration_min') > 90,
                   F.lit('LONG_TRIP')),
            F.when((F.col('trip_distance') < 0.1) & (F.col('fare_amount') > 15),
                   F.lit('GHOST_RIDE')),
            F.when((F.col('hour_of_day') < 5) & (F.col('fare_amount') > 80),
                   F.lit('LATE_SURGE')),
            F.when(F.col('passenger_count') > 6,
                   F.lit('OVERLOAD')),
        ),
        lambda x: x.isNotNull()
    ))
)

ride_anomalies = scored.filter(F.col('anomaly_score') >= ANOMALY_THRESHOLD)
print('Anomaly scoring defined.')


---
## 8 — Streaming Feature Engineering Pipeline

This section computes the **statistics per pickup zone per 5-minute window** — available in seconds.

Streaming | ML use case |
|---------------------|-------------|
| `avg_fare_5m` per zone | Surge pricing model |
| `demand_5m` per zone | Demand forecasting |
| `avg_tip_pct_5m` per zone | Driver incentive model |
| `avg_duration_5m` per zone | ETA model |
| `airport_ratio_5m` per zone | Zone classification |

### Feature Store design

```
taxi-feature-store/
  zone_features/               ← one row per (pu_location_id, 5-min window)
    year=2024/month=01/        ← Hive-partitioned for efficient time-range reads
```


In [ ]:
# ── Zone-level features: 5-minute tumbling window ─────────────────────────────
# watermark tolerates rides where the Kafka ingestion lags the event time

zone_features = (
    parsed
    .withWatermark('pickup_ts', '1 minute')  # 1 minute lag allowed for late-arriving rides
    .groupBy(
        F.window('pickup_ts', '5 minutes').alias('window'),
        'pu_location_id',
    )
    .agg(
        # ── demand features ───────────────────────────────────────────────────
        F.count('*').alias('demand_5m'),

        # ── fare features (surge pricing signals) ─────────────────────────────
        F.round(F.avg('fare_amount'),   2).alias('avg_fare_5m'),
        F.round(F.max('fare_amount'),   2).alias('max_fare_5m'),
        F.round(F.stddev('fare_amount'),2).alias('std_fare_5m'),

        # ── trip features ─────────────────────────────────────────────────────
        F.round(F.avg('trip_distance'),    2).alias('avg_distance_5m'),
        F.round(F.avg('trip_duration_min'),2).alias('avg_duration_5m'),

        # ── payment / tip features ────────────────────────────────────────────
        F.round(F.avg('tip_pct'),          2).alias('avg_tip_pct_5m'),
        F.round(F.avg('tip_amount'),       2).alias('avg_tip_5m'),

        # ── zone type features ────────────────────────────────────────────────
        F.round(F.avg('is_airport'),       3).alias('airport_ratio_5m'),

        # ── surge score: mean fare/mile across rides in this window ───────────
        F.round(F.avg('fare_per_mile'),    2).alias('avg_fare_per_mile_5m'),
    )
    .select(
        F.col('window.start').alias('window_start'),
        F.col('window.end').alias('window_end'),
        'pu_location_id',
        'demand_5m',
        'avg_fare_5m', 'max_fare_5m', 'std_fare_5m',
        'avg_distance_5m', 'avg_duration_5m',
        'avg_tip_pct_5m', 'avg_tip_5m',
        'airport_ratio_5m', 'avg_fare_per_mile_5m',
        F.current_timestamp().alias('feature_computed_at'),
    )
)

print('Zone feature schema:')
zone_features.printSchema()


### Feature Store Writer

write to Parquet, partition by date.


In [ ]:
def write_zone_features(batch_df, batch_id):
    if batch_df.isEmpty():
        return
    n = batch_df.count()
    (
        batch_df
        .withColumn('year',  F.year('window_start'))
        .withColumn('month', F.month('window_start'))
        .write
        .mode('append')
        .partitionBy('year', 'month')
        .parquet(FEATURE_STORE_PATH)
    )
    print(f'[FeatureStore] batch {batch_id}: wrote {n} zone-feature rows')


print(f'Feature store writer defined → {FEATURE_STORE_PATH}')


---
## 9 — Windowed Demand Surge Alerts

Flag zones where demand spikes sharply.


In [ ]:
demand_alerts = (
    parsed
    .withWatermark('pickup_ts', '1 minute')
    .groupBy(F.window('pickup_ts', '5 minutes'), 'pu_location_id')
    .agg(
        F.count('*').alias('ride_count'),
        F.round(F.avg('fare_amount'), 2).alias('avg_fare'),
        F.round(F.avg('trip_distance'), 2).alias('avg_distance'),
    )
    .filter(F.col('ride_count') > 10)   # zones with > 10 rides in 5 min
)
print('Demand alert stream defined.')


---
## 10 — Start All Streaming Queries

| Query | What it does | Trigger |
|-------|-------------|----------|
| `q_anomalies` | Per-ride anomaly alerts → console | every 5 s |
| `q_features` | Zone features → Parquet feature store | every 15 s |
| `q_demand` | High-demand zone alerts → console | every 10 s |


In [ ]:
shutil.rmtree(CHECKPOINT_DIR, ignore_errors=True)
shutil.rmtree(FEATURE_STORE_PATH, ignore_errors=True)

# ── Query 1: per-ride anomaly alerts ─────────────────────────────────────────
q_anomalies = (
    ride_anomalies
    .select('pickup_ts','ride_id','pu_location_id','do_location_id',
            'trip_distance','fare_amount','fare_per_mile',
            'trip_duration_min','passenger_count',
            'anomaly_score','triggered_rules')
    .writeStream
    .outputMode('update')
    .format('console')
    .option('truncate', False)
    .option('numRows', 10)
    .option('checkpointLocation', f'{CHECKPOINT_DIR}/anomalies')
    .trigger(processingTime='5 seconds')
    .queryName('ride_anomalies')
    .start()
)

# ── Query 2: zone feature engineering → feature store ────────────────────────
q_features = (
    zone_features
    .writeStream
    .outputMode('update')
    .foreachBatch(write_zone_features)
    .option('checkpointLocation', f'{CHECKPOINT_DIR}/features')
    .trigger(processingTime='15 seconds')
    .queryName('zone_feature_writer')
    .start()
)

# ── Query 3: high-demand zone alerts ─────────────────────────────────────────
q_demand = (
    demand_alerts
    .writeStream
    .outputMode('update')
    .format('console')
    .option('truncate', False)
    .option('checkpointLocation', f'{CHECKPOINT_DIR}/demand')
    .trigger(processingTime='10 seconds')
    .queryName('demand_alerts')
    .start()
)

print('Active queries:')
for q in spark.streams.active:
    print(f'  {q.name}')


---
## 11 — Monitor Streaming Progress


In [ ]:
import time
time.sleep(90)

print('=== Streaming Progress ===')
for q in spark.streams.active:
    p = q.lastProgress
    if p:
        print(f'\nQuery : {q.name}')
        print(f'  input rows/s    : {p.get("inputRowsPerSecond",  0):.1f}')
        print(f'  processed rows/s: {p.get("processedRowsPerSecond", 0):.1f}')
        print(f'  trigger ms      : {p.get("durationMs", {}).get("triggerExecution", 0)}')
        for src in p.get('sources', []):
            print(f'  kafka offsets   : {src.get("endOffset", {})}')


---
## 12 — Feature Store Explorer

Read back the zone features written during the stream 
> **Event replay:** to recompute features for a historical range, restart the producer with
> `startingOffsets='earliest'` and a date filter


In [ ]:
import os, time

# Wait until the feature writer query has flushed at least one batch
max_wait = 120   # seconds
waited   = 0
while not os.path.exists(FEATURE_STORE_PATH) and waited < max_wait:
    print(f'Waiting for feature store to be written... ({waited}s)')
    time.sleep(10)
    waited += 10

if not os.path.exists(FEATURE_STORE_PATH):
    print('Feature store not written yet — let the stream run longer (Section 11).')
else:
    features_df = spark.read.parquet(FEATURE_STORE_PATH)
    print(f'Feature rows written : {features_df.count()}')
    print(f'Unique zones         : {features_df.select("pu_location_id").distinct().count()}')
    features_df.show(5, truncate=False)


In [ ]:
# ── Top zones by demand ──────
print('=== Top 10 Zones by Demand ===')
(
    features_df
    .groupBy('pu_location_id')
    .agg(
        F.sum('demand_5m').alias('total_rides'),
        F.round(F.avg('avg_fare_5m'), 2).alias('overall_avg_fare'),
        F.round(F.avg('avg_tip_pct_5m'), 2).alias('overall_avg_tip_pct'),
        F.round(F.avg('avg_duration_5m'), 2).alias('overall_avg_duration_min'),
    )
    .orderBy(F.col('total_rides').desc())
    .limit(10)
    .show(truncate=False)
)


In [ ]:
# ── Surge candidate zones: high fare/mile (overcharging risk) ─────────────────
print('=== Surge Candidate Zones (avg fare/mile > $5) ===')
(
    features_df
    .filter(F.col('avg_fare_per_mile_5m') > 5)
    .select('pu_location_id','window_start','demand_5m',
            'avg_fare_5m','avg_fare_per_mile_5m','avg_distance_5m')
    .orderBy(F.col('avg_fare_per_mile_5m').desc())
    .limit(10)
    .show(truncate=False)
)


In [ ]:
# ── Airport zone breakdown (RatecodeID 2/3 trips) ────────────────────────────
print('=== Airport Zones (airport_ratio > 0.5) ===')
(
    features_df
    .filter(F.col('airport_ratio_5m') > 0.5)
    .select('pu_location_id','window_start','demand_5m',
            'airport_ratio_5m','avg_fare_5m','avg_distance_5m')
    .orderBy(F.col('airport_ratio_5m').desc())
    .show(truncate=False)
)


In [ ]:
# ── Feature distribution summary ─
print('=== Zone Feature Distribution ===')
features_df.select(
    'demand_5m','avg_fare_5m','avg_distance_5m',
    'avg_duration_5m','avg_tip_pct_5m','avg_fare_per_mile_5m'
).describe().show(truncate=False)


---
## 13 — Stop

In [ ]:
for q in spark.streams.active:
    q.stop()
    print(f'Stopped: {q.name}')

_stop.set()
spark.stop()
print('Done.  Tear down Kafka:  docker compose down -v')
